<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia od zera — **Lesson 12**
## 📘 **Factorizations and Other Fun** — faktoryzacje, struktury macierzy i ogólna algebra liniowa

**Cartesian School · Julia Course**  
**Autor:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Informacje o lekcji

| Pole | Wartość |
|---|---|
| Kurs | Julia od zera |
| Numer lekcji | Lesson 12 |
| Tytuł | Factorizations and Other Fun |
| Poziom | Średniozaawansowany |
| Szacowany czas | 300–420 minut |
| Wymagania | Lesson 0–11 |
| Zakres | LU, QR, Cholesky, SVD, EVD, Schur, Jordan, reuse faktoryzacji, macierze specjalne, generic linear algebra, liczby wymierne, dokładne obliczenia |
| Autor | Siergej Sobolewski |
| Prawa | © 2026 Cartesian School |

---

## Plan lekcji

1. Po co istnieją faktoryzacje.
2. Wspólne środowisko `LinearAlgebra`.
3. LU.
4. Pivoting i permutacje w LU.
5. LU do układów równań.
6. Reuse faktoryzacji.
7. Wyznacznik z LU.
8. QR.
9. QR i least squares.
10. Cholesky.
11. Warunki stosowalności Cholesky.
12. SVD.
13. Rekonstrukcja z SVD.
14. Aproksymacja niskiego rzędu.
15. EVD.
16. Eigenvalues i eigenvectors.
17. EVD a macierze symetryczne.
18. Schur decomposition.
19. Schur vs Jordan.
20. Jordan normal form — ograniczenia numeryczne.
21. Symboliczne obliczenia Jordana.
22. Macierze specjalne.
23. `Diagonal`.
24. `UpperTriangular` / `LowerTriangular`.
25. `Symmetric` / `Hermitian`.
26. `Bidiagonal`, `Tridiagonal`, `SymTridiagonal`.
27. Korzyści struktur specjalnych.
28. Generic linear algebra.
29. Liczby wymierne.
30. Dokładne układy liniowe.
31. Porównanie Float64 vs Rational.
32. Dobór faktoryzacji do problemu.
33. Typowe błędy.
34. Praktyka.
35. Mini-projekt.
36. Checkpoint.
37. Podsumowanie.


## Standard dydaktyczny Cartesian School

| Oznaczenie | Znaczenie |
|---|---|
| **Cel** | czego nauczysz się w danym fragmencie |
| **Teoria** | definicje i reguły |
| **Przykład** | minimalny, działający kod |
| **Analiza** | wyjaśnienie działania |
| **Ważne** | zasada wymagająca uwagi |
| **Typowy błąd** | częsty błąd i jego przyczyna |
| **Spróbuj sam** | mały eksperyment |
| **Praktyka** | zadanie do samodzielnego wykonania |
| **Podsumowanie** | najważniejsze wnioski |


## Cele lekcji

Po ukończeniu Lesson 12 będziesz potrafić:

- wyjaśnić, dlaczego faktoryzacje są ważniejsze niż jawne liczenie odwrotności;
- stosować LU, QR, Cholesky, SVD i EVD;
- rozumieć rolę pivotingu;
- ponownie wykorzystywać jedną faktoryzację dla wielu prawych stron;
- stosować Schur decomposition;
- rozumieć, dlaczego Jordan form jest trudna numerycznie;
- używać specjalnych struktur macierzy;
- wykonywać obliczenia na typach innych niż `Float64`;
- rozwiązywać dokładne układy na liczbach wymiernych;
- dobierać faktoryzację do struktury problemu.


## **1. Po co istnieją faktoryzacje?**

### Teoria

Faktoryzacja macierzy polega na zapisaniu macierzy jako iloczynu prostszych czynników.

Nie chodzi wyłącznie o elegancję matematyczną. Faktoryzacje są podstawą efektywnej algebry numerycznej.

Najważniejsze zastosowania:

- rozwiązywanie układów równań;
- problemy najmniejszych kwadratów;
- wyznaczniki;
- wartości własne;
- analiza rzędu;
- kompresja i aproksymacja danych;
- wielokrotne rozwiązywanie układów z tą samą macierzą.


### Główna idea

Zamiast wielokrotnie rozwiązywać:

\[
Ax=b_1,\quad Ax=b_2,\quad Ax=b_3
\]

od początku, możemy najpierw obliczyć faktoryzację:

\[
A = F
\]

a następnie używać tego samego obiektu `F` dla wielu prawych stron.


## **2. Środowisko `LinearAlgebra`**


In [ ]:
using LinearAlgebra
using Random

Random.seed!(42)

A = rand(3, 3)
x = fill(1.0, 3)
b = A * x

@show A
@show x
@show b


W kolejnych sekcjach będziemy korzystać z obiektów faktoryzacji zwracanych przez `LinearAlgebra`.


## **3. LU factorization**

### Teoria

Dla ogólnej macierzy kwadratowej LU zapisuje — z uwzględnieniem pivotingu — macierz w postaci związanej z:

\[
PA = LU
\]

gdzie:

- `P` — permutacja,
- `L` — macierz dolnotrójkątna,
- `U` — macierz górnotrójkątna.


In [ ]:
A = [
    4.0 3.0
    6.0 3.0
]

F = lu(A)

@show F
@show F.L
@show F.U
@show F.p


### Ważne

Nie zakładaj automatycznie, że `A == L*U`.

W praktycznej faktoryzacji LU często występuje pivoting, dlatego trzeba uwzględnić permutację.


## **4. Pivoting i permutacje w LU**

Pivoting poprawia stabilność numeryczną, wybierając korzystniejsze pivoty podczas eliminacji.


In [ ]:
A = [
    0.0 2.0
    1.0 3.0
]

F = lu(A)

@show F.p
@show F.L
@show F.U


### Analiza

Macierz z zerowym elementem w lewym górnym rogu wymaga zamiany kolejności wierszy, aby klasyczna eliminacja mogła przebiec poprawnie.


## **5. LU do rozwiązywania układów**


In [ ]:
A = [
    3.0 1.0
    1.0 2.0
]

b = [9.0, 8.0]

F = lu(A)
x = F \ b

@show x
@assert A * x ≈ b


### Ważne

`F \ b` wykorzystuje już policzoną faktoryzację.

To kluczowe, gdy rozwiązujemy wiele układów z tą samą macierzą `A`.


## **6. Reuse faktoryzacji**


In [ ]:
A = rand(100, 100)
F = lu(A)

b1 = rand(100)
b2 = rand(100)
b3 = rand(100)

x1 = F \ b1
x2 = F \ b2
x3 = F \ b3

@assert A * x1 ≈ b1
@assert A * x2 ≈ b2
@assert A * x3 ≈ b3


### Analiza

Najdroższa część — faktoryzacja — została policzona tylko raz.

Potem każde kolejne rozwiązanie używa gotowych czynników.


## **7. Wyznacznik z LU**

Dla macierzy kwadratowej wyznacznik można uzyskać także na podstawie faktoryzacji.


In [ ]:
A = [
    4.0 3.0
    6.0 3.0
]

F = lu(A)

@show det(A)
@show det(F)
@assert det(A) ≈ det(F)


## **8. QR factorization**

### Teoria

QR zapisuje macierz jako:

\[
A = QR
\]

gdzie:

- `Q` ma ortonormalne kolumny,
- `R` jest górnotrójkątna.


In [ ]:
A = rand(5, 3)

F = qr(A)

Q = Matrix(F.Q)
R = F.R

@show size(Q)
@show size(R)
@show Q' * Q


### Ważne

QR jest szczególnie ważne dla:

- least squares,
- ortogonalizacji,
- stabilnych metod numerycznych.


## **9. QR i least squares**


In [ ]:
A = [
    1.0 1.0
    1.0 2.0
    1.0 3.0
    1.0 4.0
]

b = [1.1, 1.9, 3.2, 4.1]

F = qr(A)
x = F \ b

@show x
@show norm(A * x - b)


### Dobra praktyka

Dla least squares nie buduj ręcznie równań normalnych `(A' * A) \ (A' * b)`, jeśli nie masz konkretnego powodu.

QR jest zwykle stabilniejszym podejściem.


## **10. Cholesky factorization**

### Teoria

Dla macierzy symetrycznej dodatnio określonej:

\[
A = LL^T
\]

lub w przypadku zespolonym:

\[
A = LL^*
\]


In [ ]:
A = [
    4.0 1.0
    1.0 3.0
]

F = cholesky(Symmetric(A))

@show F.L
@show F.U


### Korzyści

Cholesky jest bardzo efektywna dla macierzy SPD i wymaga mniej pracy niż ogólne LU.


## **11. Warunki stosowalności Cholesky**

Macierz musi być dodatnio określona.


In [ ]:
A_good = [
    4.0 1.0
    1.0 3.0
]

A_bad = [
    1.0 2.0
    2.0 1.0
]

@show isposdef(A_good)
@show isposdef(A_bad)


### Bezpieczna diagnoza błędu


In [ ]:
cholesky_failed = try
    cholesky(Symmetric(A_bad))
    false
catch
    true
end

@show cholesky_failed


## **12. SVD — Singular Value Decomposition**

### Teoria

SVD rozkłada macierz:

\[
A = U\Sigma V^*
\]

i działa także dla macierzy prostokątnych.


In [ ]:
A = rand(5, 3)

F = svd(A)

@show F.S
@show size(F.U)
@show size(F.Vt)


### SVD jest fundamentalne dla:

- analizy rzędu,
- pseudoodwrotności,
- kompresji,
- PCA,
- aproksymacji niskiego rzędu,
- diagnozowania problemów źle uwarunkowanych.


## **13. Rekonstrukcja z SVD**


In [ ]:
A = rand(5, 3)
F = svd(A)

A_reconstructed = F.U * Diagonal(F.S) * F.Vt

@show norm(A - A_reconstructed)
@assert A ≈ A_reconstructed


## **14. Aproksymacja niskiego rzędu**

Zatrzymując tylko największe singular values, możemy otrzymać przybliżenie macierzy.


In [ ]:
A = rand(20, 10)
F = svd(A)

k = 3

A_k = F.U[:, 1:k] * Diagonal(F.S[1:k]) * F.Vt[1:k, :]

@show rank(A_k)
@show norm(A - A_k)


### Analiza

To podstawa wielu metod kompresji i redukcji wymiarowości.


## **15. EVD — eigendecomposition**

### Teoria

Dla macierzy kwadratowej szukamy:

\[
Av = \lambda v
\]

gdzie `λ` jest wartością własną, a `v` wektorem własnym.


In [ ]:
A = [
    2.0 1.0
    1.0 2.0
]

E = eigen(A)

@show E.values
@show E.vectors


## **16. Sprawdzenie pary własnej**


In [ ]:
λ = E.values[1]
v = E.vectors[:, 1]

@show A * v
@show λ * v
@assert A * v ≈ λ * v


## **17. EVD a macierze symetryczne**

Dla macierzy rzeczywistej symetrycznej:

- wartości własne są rzeczywiste,
- wektory własne można wybrać ortonormalne.


In [ ]:
A = Symmetric([
    4.0 1.0
    1.0 2.0
])

E = eigen(A)

@show E.values
@show E.vectors' * E.vectors


### Dobra praktyka

Jeżeli macierz jest naprawdę symetryczna, przekazanie `Symmetric(A)` pozwala użyć właściwszych algorytmów.


## **18. Schur decomposition**

### Teoria

Rozkład Schura zapisuje macierz w postaci:

\[
A = Q T Q^*
\]

gdzie `Q` jest unitarną/ortogonalną macierzą, a `T` ma postać trójkątną lub quasi-trójkątną.


In [ ]:
A = rand(4, 4)

S = schur(A)

@show S.values
@show size(S.Z)
@show size(S.T)


### Zastosowania

Schur decomposition jest ważne w stabilnych algorytmach wartości własnych i funkcji macierzy.


## **19. Schur vs Jordan**

Jordan normal form jest ważna teoretycznie, ale rozkład Schura jest zwykle znacznie bardziej odpowiedni do obliczeń numerycznych.

Dlaczego?

- Schur jest numerycznie stabilniejszy,
- Jordan form jest bardzo wrażliwa na małe perturbacje,
- w praktyce floatowej nawet minimalny szum może zmienić strukturę bloków Jordana.


## **20. Jordan normal form — ograniczenia numeryczne**

### Ważne

`LinearAlgebra` nie udostępnia ogólnej funkcji numerycznej `jordan(A)` jako standardowego narzędzia.

To nie jest brak przypadkowy. Jordan form jest trudna i niestabilna numerycznie.

Do pracy numerycznej zwykle preferujemy:

- Schur,
- eigen,
- SVD.


### Kiedy Jordan ma sens?

- w algebrze symbolicznej,
- w analizie teoretycznej,
- w dokładnej arytmetyce,
- w dydaktyce struktur operatorów liniowych.


## **21. Symboliczna postać Jordana — przykład koncepcyjny**

Pakiety zewnętrzne mogą oferować narzędzia symboliczne lub dokładne, ale API zależy od konkretnego pakietu i jego wersji.

Dlatego w kursie nie instalujemy automatycznie `JordanForm.jl`, `Symbolics.jl` ani podobnych pakietów.


In [ ]:
# Przykład koncepcyjny — wymaga zewnętrznego pakietu:
#
# using Pkg
# Pkg.add("JordanForm")
#
# Następnie należy używać API zgodnego z aktualną dokumentacją pakietu.


### Ważne

Nie mieszaj:

- obliczeń numerycznych `Float64`,
- obliczeń symbolicznych,
- dokładnej arytmetyki wymiernej.

To różne modele obliczeń i mają inne własności.


## **22. Specjalne struktury macierzy**

Julia potrafi przechowywać informację o strukturze macierzy w samym typie.


### Dlaczego to ważne?

Jeżeli wiemy, że macierz jest:

- diagonalna,
- trójkątna,
- symetryczna,
- hermitowska,
- trójdiagonalna,

nie musimy traktować jej jak dowolnej gęstej `Matrix`.


## **23. `Diagonal`**


In [ ]:
D = Diagonal([1.0, 2.0, 3.0])

@show D
@show typeof(D)


In [ ]:
x = [10.0, 20.0, 30.0]

@show D * x


### Analiza

`Diagonal` przechowuje tylko przekątną, a nie wszystkie zera poza nią.


## **24. `UpperTriangular` i `LowerTriangular`**


In [ ]:
A = [
    1.0 2.0 3.0
    4.0 5.0 6.0
    7.0 8.0 9.0
]

U = UpperTriangular(A)
L = LowerTriangular(A)

@show U
@show L


## **25. `Symmetric` i `Hermitian`**


In [ ]:
A = [
    2.0 1.0
    1.0 3.0
]

S = Symmetric(A)

@show S
@show typeof(S)


In [ ]:
Z = [
    2 + 0im  1 + 2im
    1 - 2im  4 + 0im
]

H = Hermitian(Z)

@show H


## **26. `Bidiagonal`, `Tridiagonal`, `SymTridiagonal`**


In [ ]:
d = [2.0, 2.0, 2.0, 2.0]
e = [-1.0, -1.0, -1.0]

T = SymTridiagonal(d, e)

@show T
@show typeof(T)


### Zastosowania

Macierze trójdiagonalne pojawiają się m.in. w:

- dyskretyzacji równań różniczkowych,
- metodach elementów skończonych,
- modelach łańcuchowych,
- zagadnieniach własnych.


## **27. Korzyści struktur specjalnych**

Specjalna struktura może oznaczać:

- mniejsze zużycie pamięci,
- mniej operacji,
- wyspecjalizowany solver,
- czytelniejszy kontrakt matematyczny.


### Typowy błąd

Nie konwertuj bez potrzeby:

```julia
Matrix(D)
```

jeżeli algorytm potrafi pracować bezpośrednio z `Diagonal`.


## **28. Generic linear algebra**

### Teoria

Jedna z mocnych stron Julii: wiele algorytmów algebry liniowej działa dla typów innych niż `Float64`.

Możemy pracować m.in. z:

- `Float32`,
- `BigFloat`,
- `Complex`,
- liczbami wymiernymi,
- wybranymi własnymi typami numerycznymi.


## **29. Liczby wymierne**

W Julia liczby wymierne zapisujemy:


In [ ]:
a = 1 // 3
b = 2 // 5

@show a
@show b
@show a + b
@show typeof(a)


### Ważne

`1//3` reprezentuje dokładnie jedną trzecią, a nie przybliżenie `Float64`.


## **30. Dokładny układ liniowy na Rational**


In [ ]:
A = [
    1//1  1//2
    1//3  1//1
]

b = [1//1, 1//1]

x = A \ b

@show x
@show A * x
@assert A * x == b


### Analiza

Wynik jest dokładny w arytmetyce wymiernej — nie ma błędu zaokrąglenia typowego dla `Float64`.


## **31. Float64 vs Rational**


In [ ]:
A_float = Float64[
    1.0 0.5
    1/3 1.0
]

b_float = [1.0, 1.0]

x_float = A_float \ b_float

A_rat = [
    1//1 1//2
    1//3 1//1
]

b_rat = [1//1, 1//1]

x_rat = A_rat \ b_rat

@show x_float
@show x_rat


### Ważne

Dokładność ma koszt.

Arytmetyka wymierna może być znacznie wolniejsza i generować większe liczby w licznikach/mianownikach.

Dobór typu liczbowego zależy od celu obliczeń.


## **32. Jak dobrać faktoryzację?**

| Problem | Preferowane narzędzie |
|---|---|
| ogólny układ kwadratowy | LU / `A \ b` |
| macierz SPD | Cholesky |
| least squares | QR / `A \ b` |
| analiza rzędu | SVD |
| kompresja niskiego rzędu | SVD |
| wartości własne symetryczne | `eigen(Symmetric(A))` |
| ogólna analiza wartości własnych | Schur / eigen |
| teoria bloków Jordana | algebra symboliczna / dokładna |


### Dobra praktyka

Najpierw wykorzystaj strukturę matematyczną problemu, dopiero potem wybieraj algorytm.

Ogólny solver zawsze działa „bardziej ogólnie”, ale nie zawsze jest najlepszym wyborem.


## **33. Typowe błędy**

| Błąd | Problem | Poprawne podejście |
|---|---|---|
| `inv(A)*b` | niepotrzebna odwrotność | `A\b` lub faktoryzacja |
| ignorowanie pivotingu LU | błędna rekonstrukcja | uwzględnij permutację |
| Cholesky dla macierzy nie-SPD | algorytm nie ma warunków | `isposdef`, właściwa faktoryzacja |
| normal equations w least squares | gorsza kondycja | QR / `A\b` |
| SVD traktowane jak eigen | inne pojęcia | rozróżniaj zastosowania |
| oczekiwanie stabilnej Jordan form dla Float64 | niestabilność numeryczna | Schur / symbolic |
| zamiana struktur specjalnych na gęste | strata informacji i pamięci | zachowaj specjalny typ |
| oczekiwanie szybkości Rational jak Float64 | exact arithmetic ma koszt | dobierz typ do celu |


## **34. Praktyka**

### Zadanie 34.1 — wartości własne

Dla macierzy:

```julia
A = [2.0 1.0; 1.0 2.0]
```

oblicz wartości własne.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 34.1


In [ ]:
A = [2.0 1.0; 1.0 2.0]

λ = eigvals(Symmetric(A))

@show λ


### Zadanie 34.2 — diagonalna macierz wartości własnych

Zbuduj `Diagonal(λ)`.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 34.2


In [ ]:
A = [2.0 1.0; 1.0 2.0]
λ = eigvals(Symmetric(A))
Λ = Diagonal(λ)

@show Λ


### Zadanie 34.3 — macierz dolnotrójkątna

Utwórz `LowerTriangular` z dowolnej macierzy `3×3`.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 34.3


In [ ]:
A = reshape(1:9, 3, 3)
L = LowerTriangular(A)

@show L


### Zadanie 34.4 — wybór rozkładu

Dobierz faktoryzację:

1. ogólna macierz kwadratowa,
2. macierz SPD,
3. least squares,
4. analiza niskiego rzędu.


In [ ]:
# Odpowiedź tekstowa / komentarze:
#
# 1.
# 2.
# 3.
# 4.


### Przykładowe rozwiązanie 34.4

1. LU,
2. Cholesky,
3. QR,
4. SVD.


### Zadanie 34.5 — reuse faktoryzacji

Dla jednej macierzy `A` rozwiąż trzy układy z różnymi prawymi stronami, licząc faktoryzację tylko raz.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 34.5


In [ ]:
A = rand(5, 5)
F = lu(A)

B = rand(5, 3)
X = F \ B

@assert A * X ≈ B


## **35. Mini-projekt — solver dobierający faktoryzację**

### Cel

Zbudujemy prostą funkcję dydaktyczną, która wykorzysta informację o strukturze macierzy.


In [ ]:
function solve_structured(A::AbstractMatrix, b::AbstractVecOrMat)
    size(A, 1) == size(A, 2) || return qr(A) \ b

    if issymmetric(A) && isposdef(A)
        return cholesky(Symmetric(A)) \ b
    end

    return lu(A) \ b
end


### Test 1 — SPD


In [ ]:
A1 = [
    4.0 1.0
    1.0 3.0
]
b1 = [1.0, 2.0]

x1 = solve_structured(A1, b1)

@assert A1 * x1 ≈ b1


### Test 2 — ogólna macierz


In [ ]:
A2 = [
    1.0 2.0
    3.0 5.0
]
b2 = [1.0, 1.0]

x2 = solve_structured(A2, b2)

@assert A2 * x2 ≈ b2


### Test 3 — least squares


In [ ]:
A3 = rand(6, 3)
b3 = rand(6)

x3 = solve_structured(A3, b3)

@show norm(A3 * x3 - b3)


### Analiza

To nie jest kompletny solver produkcyjny, ale dobrze pokazuje ważną zasadę:

> struktura macierzy powinna wpływać na wybór algorytmu.


## **36. Checkpoint końcowy**

Odpowiedz bez uruchamiania kodu:

1. Po co stosuje się faktoryzacje?
2. Co oznacza `PA = LU`?
3. Dlaczego LU używa pivotingu?
4. Co daje reuse faktoryzacji?
5. Do jakich problemów szczególnie nadaje się QR?
6. Jakie warunki musi spełniać macierz dla Cholesky?
7. Co zwraca SVD?
8. Do czego służy aproksymacja niskiego rzędu?
9. Co opisuje EVD?
10. Dlaczego `Symmetric(A)` jest użyteczne?
11. Co daje Schur decomposition?
12. Dlaczego Jordan form jest problematyczna numerycznie?
13. Po co używać `Diagonal` zamiast pełnej macierzy?
14. Co daje `SymTridiagonal`?
15. Czym jest generic linear algebra?
16. Czym różni się `1//3` od `1/3`?
17. Dlaczego Rational może dawać wynik dokładny?
18. Dlaczego exact arithmetic jest droższa?
19. Jak dobrać faktoryzację do macierzy SPD?
20. Jak dobrać faktoryzację do least squares?


## **37. Podsumowanie lekcji**

Najważniejsze zasady Lesson 12:

1. Faktoryzacje są fundamentem praktycznej algebry numerycznej.
2. LU jest naturalnym wyborem dla ogólnych układów kwadratowych.
3. Pivoting jest częścią stabilnej implementacji LU.
4. Faktoryzację warto ponownie wykorzystywać dla wielu prawych stron.
5. QR jest kluczowa dla least squares.
6. Cholesky jest bardzo efektywna dla macierzy SPD.
7. SVD jest jednym z najbardziej uniwersalnych narzędzi analizy macierzy.
8. EVD opisuje wartości i wektory własne macierzy kwadratowej.
9. Schur decomposition jest ważnym, stabilnym narzędziem numerycznym dla problemów własnych.
10. Jordan form ma przede wszystkim znaczenie teoretyczne i symboliczne, nie jako standardowa metoda numeryczna `Float64`.
11. Specjalne typy macierzy zachowują strukturę matematyczną i mogą poprawić wydajność.
12. Julia wspiera generic linear algebra na wielu typach liczbowych.
13. Liczby wymierne pozwalają wykonywać dokładne obliczenia algebraiczne.
14. Dobór faktoryzacji powinien wynikać ze struktury problemu, a nie z przyzwyczajenia.


## Źródła do dalszego czytania

- Julia Standard Library — LinearAlgebra
- LinearAlgebra — Factorizations
- LinearAlgebra — Structured Matrices
- LinearAlgebra — Eigenvalues
- LinearAlgebra — Singular Value Decomposition
- LinearAlgebra — Schur Factorization
- Julia Manual — Mathematical Operations
- Julia Manual — Types


[← Lesson 11 — Linear Algebra in Julia](Lesson_11_Linear_Algebra_in_Julia_Cartesian_School_PL.ipynb)  
[Spis treści](../README.pl.md)  
[Lesson 13 — Numerical Computing →](Lesson_13_Numerical_Computing_Julia_Cartesian_School_PL.ipynb)
